# Project Two Query Unit Tests
**Jacobsin Enriquez | CS 340 Client/Server Development**

Unit tests for the three rescue-type filter queries defined by the Rescue Type and Preferred Dog Breeds table
in the Dashboard Specifications Document. Each query is run through the Project One CRUD module, and every
returned record is checked against its breed, sex, and age rules. Breed matching is tested two ways
(exact name vs. name contained in the breed field) so the dashboard can use the better fit for the data.

In [ ]:
# Pull in the custom CRUD module built for database interaction
# Ensure CRUD_Python_Module.py is in the local directory
import re
import pandas as pd
from CRUD_Python_Module import AnimalShelter

# Authenticate using the restricted aacuser credentials configured in Module Three
shelter = AnimalShelter("aacuser", "password123")

In [ ]:
# Rescue profiles transcribed from the Dashboard Specifications Document
# Each entry holds preferred breeds, preferred sex, and training age range in weeks
RESCUE_PROFILES = {
    "Water Rescue": {
        "breeds": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"],
        "sex": "Intact Female",
        "min_weeks": 26, "max_weeks": 156,
    },
    "Mountain or Wilderness Rescue": {
        "breeds": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog",
                   "Siberian Husky", "Rottweiler"],
        "sex": "Intact Male",
        "min_weeks": 26, "max_weeks": 156,
    },
    "Disaster or Individual Tracking": {
        "breeds": ["Doberman Pinscher", "German Shepherd", "Golden Retriever",
                   "Bloodhound", "Rottweiler"],
        "sex": "Intact Male",
        "min_weeks": 20, "max_weeks": 300,
    },
}


def build_query(profile, mode="exact"):
    """
    Assemble a MongoDB query document from a rescue profile.
    mode="exact"    matches the breed name exactly as listed (e.g. "German Shepherd")
    mode="contains" also matches mixes containing the name (e.g. "German Shepherd Mix")
    """
    if mode == "exact":
        breed_filter = {"$in": profile["breeds"]}
    else:
        # Escaped regex patterns match the breed name anywhere in the field
        breed_filter = {"$in": [re.compile(re.escape(b)) for b in profile["breeds"]]}

    return {
        "animal_type": "Dog",
        "breed": breed_filter,
        "sex_upon_outcome": profile["sex"],
        "age_upon_outcome_in_weeks": {"$gte": profile["min_weeks"], "$lte": profile["max_weeks"]},
    }

In [ ]:
# Unit test: run each query and verify every returned record meets its profile rules
def validate(name, profile, mode):
    results = shelter.read(build_query(profile, mode))
    failures = []
    for doc in results:
        breed = doc.get("breed", "")
        breed_ok = (breed in profile["breeds"]) if mode == "exact" \
            else any(b in breed for b in profile["breeds"])
        sex_ok = doc.get("sex_upon_outcome") == profile["sex"]
        age = doc.get("age_upon_outcome_in_weeks", -1)
        age_ok = profile["min_weeks"] <= age <= profile["max_weeks"]
        type_ok = doc.get("animal_type") == "Dog"
        if not (breed_ok and sex_ok and age_ok and type_ok):
            failures.append(doc.get("animal_id"))

    if not results:
        status = "WARNING - no records returned"
    elif failures:
        status = f"FAIL - {len(failures)} records break the rules: {failures[:5]}"
    else:
        status = "PASS - every record matches breed, sex, age, and type"
    print(f"{name} [{mode}]: {len(results)} records | {status}")
    return results


exact_results = {}
for name, profile in RESCUE_PROFILES.items():
    exact_results[name] = validate(name, profile, "exact")

In [ ]:
# Comparison: exact breed match vs. "contains" match (includes mixes)
# Breed breakdown shows which matching style gives the client a more useful candidate list
for name, profile in RESCUE_PROFILES.items():
    contains_results = validate(name, profile, "contains")
    print("  Breeds returned with 'contains' matching:")
    if contains_results:
        counts = pd.Series([d["breed"] for d in contains_results]).value_counts()
        print(counts.head(10).to_string())
    print()

In [ ]:
# Reset test: the unfiltered query must return the full collection
total = shelter.collection.count_documents({})
all_records = shelter.read({})
print(f"Reset [all records]: {len(all_records)} of {total} | "
      f"{'PASS' if len(all_records) == total else 'FAIL'}")

# Column layout check: after dropping _id, positions 4, 9, 13, 14 should hold breed, name,
# latitude, longitude (the starter map code read these by position; the dashboard now reads by name)
df = pd.DataFrame.from_records(all_records).drop(columns=["_id"])
expected = {4: "breed", 9: "name", 13: "location_lat", 14: "location_long"}
for pos, col in expected.items():
    print(f"Column {pos}: {df.columns[pos]} | {'PASS' if df.columns[pos] == col else 'FAIL'}")